In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr # oh yeah!
from scraper import fetch_website_contents,fetch_website_links
import json
from IPython.display import display,Markdown,update_display

# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")
system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    """

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

user_prompt("https://www.linkedin.com")

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling gpt nano model")

    response = openai.chat.completions.create(
        model = "gpt-5-nano",
        messages = [
            {"role" : "system" , "content": system_prompt},
            {"role" : "user", "content" : user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    return links


def get_page_and_relevant_links(url):
    content = fetch_website_contents(url)
    print(f"\n Fetch content of the company {url}\n")

    links = select_relevant_links(url)
    result = f" Landing Page:\n\n{content}\n Relevant Links:\n"

    for link in links['links']:
        result += f"\n\nLink: {link['type']}\n\n"
        result += fetch_website_contents(link["url"])
    return result


brochure_prompt_system = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.

"""

def user_brochure_prompt(companyname,url):
    user_brochure_prompt = f"""Please fetch the contents of the {companyname}
    Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_brochure_prompt += get_page_and_relevant_links(url)
    user_brochure_prompt += user_brochure_prompt[:5_000]
    return user_brochure_prompt

user_brochure_prompt("Hugging Face","https://huggingface.co")

def create_brochure(company_name,url):
    stream = openai.chat.completions.create(model = "gpt-4.1-mini",
    messages = [
        { "role" : "system" , "content":brochure_prompt_system},
        { "role" : "user", "content": user_brochure_prompt(company_name,url)}
    ],
    stream = True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id = display_handle.display_id)

create_brochure("HuggingFace", "https://huggingface.co")




name_input = gr.Textbox(label = "Company Name")
company_url = gr.Textbox(label = "Company landing page url")
name_output = gr.Markdown(label ="Response")

view = gr.Interface(
    fn = create_brochure,
    title = "Get Brochure of a company",
    inputs = [name_input,company_url],
    outputs = [name_output],
    examples = [
        ["Hugging Face", "https://huggingface.co"],
        ["Edward Donner", "https://edwarddonner.com"]
    ],
    flagging_mode = "never"
    ).launch(inbrowser=True, auth=("suppi", "akella"))